# Stage A — Distributional coverage of sat ↔ MERRA-2 training pairs (v3.4.0)

Under v3.4.0 §7.4.1, the soft calibration `MERRA-2 = α · sat + β` is fit per
(sensor, region, season) on **sat ↔ MERRA-2** pairs assembled from Stage A2
gridded slots over **Sep 2022 – Dec 2024**.  This notebook checks whether
those training pairs are representative — i.e. whether they span the satellite
AOD distribution the production fusion will actually encounter.  If they don't,
the (α, β) is fit on a non-representative sample and the §7.4.1 guard rail
(N ≥ 100, α ∈ [0.5, 2.0], |β| ≤ 0.2) is the only thing standing between bad
input and bad weights.

Coverage has two components and **both** must hold:
1. **Range coverage** — training sat values span the production sat values in
   the same (sensor, region, season) stratum.
2. **Density coverage** — training pairs are distributed across that range,
   not clumped in one zone.

Diagnostics (per `sensor × region × season` stratum):

| # | What | Pass criterion | Why it matters |
|---|---|---|---|
| 1 | Pair-count vs `SOFT_CAL_MIN_PAIRS` | N ≥ 100 | Below this the §7.4.1 guard rail routes to `'none'` automatically. |
| 2 | Training-sat CDF vs held-out-sat CDF (KS) | KS ≤ 0.10 | Production-window distribution must look like training-window distribution. |
| 3 | 95th / 99th percentile envelope train vs held-out | overlap | If held-out tails extend beyond training, the linear extrapolation in §7.4.1 is suspect. |
| 4 | Decile bin counts of training sat AOD | no empty deciles in (0, p99) | Empty deciles mean the fit is interpolating across an unobserved zone. |
| 5 | sat-vs-MERRA-2 scatter with fitted line | visual sanity | The §7.4.1 fit (Ding et al. 2025 form). |
| 6 | Training-sat CDF vs AERONET CDF at the two AERONET cells | broadly overlapping | Sanity that the train regime covers the regime §8 will validate against. |

**v3.4.0 note** — the old (v3.3.x) notebook compared sat ↔ AERONET matched-pair
coverage to the full AERONET record.  That question doesn't apply in v3.4.0
because AERONET is no longer a training input (§7.4 preamble: "AERONET is
reserved entirely for held-out validation").  Check #6 below preserves the
AERONET comparison as a diagnostic only — it is **not** pass/fail.


## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from config import (
    TRAIN_START, TRAIN_END, TEST_START, TEST_END,
    SOFT_CAL_MIN_PAIRS, REGIONS, SEASONS, DRY_MONTHS,
    AERONET_SITES,
)
from bias_correction import collect_sat_merra2_pairs

SENSORS = ('himawari_l2', 'himawari_l3', 'viirs_snpp', 'viirs_noaa20', 'modis_maiac')

# Sampling stride for the gridded-slot walk.  Every cell is independent so a
# stride of 4 thins I/O 4× without biasing distribution estimates; bump to 1 for
# a final figure.
TRAIN_STRIDE = 4
TEST_STRIDE  = 4

# Pass-criteria thresholds
KS_MAX           = 0.10   # train-vs-heldout CDF KS (§2)
P99_OVERSHOOT_MAX = 0.10  # held-out p99 may exceed training p99 by ≤ 10%

AERONET_FULL_DIR = Path('/home/slow_data/Air_Quality/AERONET/process/aod_550/10')

print(f'Training window : {TRAIN_START} → {TRAIN_END} (stride {TRAIN_STRIDE})')
print(f'Held-out window : {TEST_START} → {TEST_END} (stride {TEST_STRIDE})')
print(f'Guard rail floor: N ≥ {SOFT_CAL_MIN_PAIRS}')
print(f'AERONET diagnostic: {AERONET_FULL_DIR}')

## 1. Collect sat ↔ MERRA-2 pairs on the training and held-out windows

`collect_sat_merra2_pairs` walks Stage A2 gridded NetCDFs over the requested
window, reads MERRA-2 hourly slices resampled to 0.05° for the matching slot,
and emits per (region, season) arrays of `(sat, MERRA-2)` for one sensor.
Run it twice — once on each window — for every sensor.

*This is the slowest cell in the notebook.  Stride 4 keeps it tractable.*

In [ ]:
train_by_sensor = {}
test_by_sensor  = {}

for sensor in SENSORS:
    print(f'\n  {sensor}: training pairs')
    train_by_sensor[sensor] = collect_sat_merra2_pairs(
        sensor=sensor, train_start=TRAIN_START, train_end=TRAIN_END,
        sample_every_n_slots=TRAIN_STRIDE,
    )
    print(f'  {sensor}: held-out pairs')
    test_by_sensor[sensor] = collect_sat_merra2_pairs(
        sensor=sensor, train_start=TEST_START, train_end=TEST_END,
        sample_every_n_slots=TEST_STRIDE,
    )

rows = []
for sensor in SENSORS:
    for region in REGIONS:
        for season in SEASONS:
            tr = train_by_sensor[sensor].get((region, season))
            te = test_by_sensor[sensor].get((region, season))
            n_tr = int(tr['sat'].size) if tr is not None else 0
            n_te = int(te['sat'].size) if te is not None else 0
            rows.append({
                'sensor': sensor, 'region': region, 'season': season,
                'n_train': n_tr, 'n_heldout': n_te,
            })
counts = pd.DataFrame(rows)
print()
print('Pair counts per stratum (n_train uses TRAIN_STRIDE; multiply by stride for true N):')
counts.round(0)

## 2. Range / density diagnostics per stratum

Three numbers per (sensor, region, season) stratum:
* `KS` — two-sample Kolmogorov-Smirnov statistic on the sat AOD distributions
  of training vs held-out pairs.  Small KS = same regime.
* `p99_ratio` — held-out 99th percentile / training 99th percentile of sat AOD.
  Values > 1.10 mean the held-out window saw higher AOD than training — the
  linear (α, β) is extrapolating into an unobserved tail.
* `empty_deciles` — number of training deciles (within `[0, p99_train]`) with
  density < 0.5 × the uniform expectation.  Empty deciles mean the fit
  bridges across an unobserved zone.

In [ ]:
def _decile_gaps(sat: np.ndarray, p99: float) -> int:
    if sat.size < 100 or not np.isfinite(p99) or p99 <= 0:
        return -1
    sat_in = sat[(sat >= 0) & (sat <= p99)]
    if sat_in.size < 100:
        return -1
    edges = np.linspace(0, p99, 11)
    counts, _ = np.histogram(sat_in, bins=edges)
    uniform = sat_in.size / 10
    return int((counts < 0.5 * uniform).sum())

diag_rows = []
for sensor in SENSORS:
    for region in REGIONS:
        for season in SEASONS:
            tr = train_by_sensor[sensor].get((region, season))
            te = test_by_sensor[sensor].get((region, season))
            if tr is None or te is None or tr['sat'].size < 10 or te['sat'].size < 10:
                continue
            sat_tr = tr['sat']
            sat_te = te['sat']
            ks_stat, _ = stats.ks_2samp(sat_tr, sat_te)
            p99_tr = float(np.percentile(sat_tr, 99))
            p99_te = float(np.percentile(sat_te, 99))
            p95_tr = float(np.percentile(sat_tr, 95))
            p95_te = float(np.percentile(sat_te, 95))
            gaps   = _decile_gaps(sat_tr, p99_tr)
            diag_rows.append({
                'sensor': sensor, 'region': region, 'season': season,
                'n_train': int(sat_tr.size), 'n_heldout': int(sat_te.size),
                'KS': float(ks_stat),
                'p95_train': p95_tr, 'p95_heldout': p95_te,
                'p99_train': p99_tr, 'p99_heldout': p99_te,
                'p99_ratio': p99_te / p99_tr if p99_tr > 0 else np.nan,
                'empty_deciles': gaps,
            })
diag = pd.DataFrame(diag_rows)

if not diag.empty:
    diag['KS_pass']      = diag['KS']        <= KS_MAX
    diag['p99_pass']     = diag['p99_ratio'] <= 1.0 + P99_OVERSHOOT_MAX
    diag['density_pass'] = diag['empty_deciles'] <= 1
    diag['coverage_pass'] = diag['KS_pass'] & diag['p99_pass'] & diag['density_pass']
    print('Per-stratum coverage diagnostics (apply-eligible strata only):')
    print(diag.round(3).to_string(index=False))
    print()
    fail = diag[~diag['coverage_pass']]
    if not fail.empty:
        print('Strata with weak coverage — flag in §10 limitations:')
        print(fail[['sensor', 'region', 'season', 'KS', 'p99_ratio', 'empty_deciles']]
              .round(3).to_string(index=False))
else:
    print('No strata with paired data — confirm Stage A2 grids are populated.')

## 3. CDF overlay — train vs held-out sat AOD per stratum

Visual companion to §2's KS / p99 / decile numbers.  One sensor per figure;
set `PLOT_SENSORS` to restrict the set.

In [ ]:
PLOT_SENSORS = ('himawari_l2', 'modis_maiac')   # () for all

def _plot_cdf(sensor: str) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharex=True, sharey=True)
    fig.suptitle(f'{sensor} — sat AOD CDF (train vs held-out)', fontsize=11)
    for i, season in enumerate(SEASONS):
        for j, region in enumerate(REGIONS):
            ax = axes[i, j]
            tr = train_by_sensor[sensor].get((region, season))
            te = test_by_sensor[sensor].get((region, season))
            for d, label, color in ((tr, 'train', 'C0'), (te, 'held-out', 'C1')):
                if d is None or d['sat'].size == 0:
                    continue
                x = np.sort(d['sat'])
                y = np.arange(1, x.size + 1) / x.size
                ax.plot(x, y, color=color, lw=1, label=f'{label} (N={x.size})')
            ax.set_xlim(0, 2); ax.set_ylim(0, 1)
            ax.set_title(f'{region} · {season}', fontsize=9)
            if i == 1: ax.set_xlabel('sat AOD')
            if j == 0: ax.set_ylabel('CDF')
            ax.legend(fontsize=7, loc='lower right')
    fig.tight_layout()
    plt.show()

for s in (PLOT_SENSORS or SENSORS):
    _plot_cdf(s)

## 4. sat-vs-MERRA-2 scatter with §7.4.1 linear-fit overlay

Acid test that the linear (α, β) form is honest for the high-density region
of each stratum.  Training pairs (blue) are overlaid with the OLS line in
red.  A sensor whose plume looks heavily curved tells you the linear form
is fighting structure (in which case the §7.4.1 guard rail should be
tripping).

In [ ]:
def _ols(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    sx, sy = x.mean(), y.mean()
    sxx = float(np.sum((x - sx) ** 2))
    if sxx < 1e-12:
        return float('nan'), float('nan')
    a = float(np.sum((x - sx) * (y - sy)) / sxx)
    return a, float(sy - a * sx)

def _plot_fit(sensor: str) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharex=True, sharey=True)
    fig.suptitle(f'{sensor} — sat vs MERRA-2 (training window)', fontsize=11)
    for i, season in enumerate(SEASONS):
        for j, region in enumerate(REGIONS):
            ax = axes[i, j]
            tr = train_by_sensor[sensor].get((region, season))
            if tr is None or tr['sat'].size < 10:
                ax.set_visible(False); continue
            sat, ref = tr['sat'], tr['merra2']
            ax.scatter(sat, ref, s=1, alpha=0.1, color='C0')
            a, b = _ols(sat.astype(np.float64), ref.astype(np.float64))
            xs = np.linspace(0, 2, 50)
            if np.isfinite(a):
                ax.plot(xs, a * xs + b, 'r-', lw=1.2,
                        label=f'α={a:.2f} β={b:+.2f} N={sat.size}')
            ax.plot([0, 2], [0, 2], 'k--', lw=0.5)
            ax.set_xlim(0, 2); ax.set_ylim(0, 2)
            ax.set_title(f'{region} · {season}', fontsize=9)
            if i == 1: ax.set_xlabel('sat AOD')
            if j == 0: ax.set_ylabel('MERRA-2 AOD')
            ax.legend(fontsize=7, loc='upper left')
    fig.tight_layout()
    plt.show()

for s in (PLOT_SENSORS or SENSORS):
    _plot_fit(s)

## 5. AERONET regime overlap (diagnostic, not pass/fail)

Does the training-pair sat AOD distribution span the regime that AERONET
validation will probe in §8.1.1?  If training is consistently lower than
AERONET, §8.1.1 will encounter many production cells whose AOD lies in a
training-blind zone.

This is **diagnostic-only** — AERONET is reserved for held-out validation
per the v3.4.0 §7.4 preamble.  No training table is rebuilt from this check.

In [ ]:
def load_full_aeronet(site: str) -> pd.DataFrame:
    path = AERONET_FULL_DIR / f'{site}.csv'
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, parse_dates=['datetime'])
    df = df[(df['datetime'] >= pd.Timestamp(TRAIN_START)) &
            (df['datetime'] <= pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1))]
    df = df[df['aod_550'].notna() & (df['aod_550'] >= 0)].copy()
    df['season'] = np.where(df['datetime'].dt.month.isin(DRY_MONTHS), 'dry', 'wet')
    df['site']   = site
    return df

aer_sites = sorted(AERONET_SITES)
aer_full = pd.concat([load_full_aeronet(s) for s in aer_sites], ignore_index=True)

site_to_region = {'NGHIA_DO': 'north', 'Bac_Lieu': 'south'}

rows = []
for site in aer_sites:
    region = site_to_region.get(site)
    if region is None:
        continue
    aer_site = aer_full[aer_full['site'] == site]
    for season in SEASONS:
        aer_v = aer_site.loc[aer_site['season'] == season, 'aod_550'].to_numpy()
        if aer_v.size < 10:
            continue
        for sensor in SENSORS:
            tr = train_by_sensor[sensor].get((region, season))
            if tr is None or tr['sat'].size < 10:
                continue
            ks_stat, _ = stats.ks_2samp(tr['sat'], aer_v)
            rows.append({
                'sensor': sensor, 'site': site, 'region': region, 'season': season,
                'n_train_sat': int(tr['sat'].size), 'n_aeronet': int(aer_v.size),
                'p95_train_sat': float(np.percentile(tr['sat'], 95)),
                'p95_aeronet':   float(np.percentile(aer_v, 95)),
                'KS_train_vs_aeronet': float(ks_stat),
            })
aer_diag = pd.DataFrame(rows)
if not aer_diag.empty:
    print('Training-sat vs full-AERONET (training-window) distribution overlap:')
    print(aer_diag.round(3).to_string(index=False))
    print()
    big_gap = aer_diag[aer_diag['KS_train_vs_aeronet'] > 0.25]
    if not big_gap.empty:
        print('Strata with KS > 0.25 against AERONET — note in §10 (diagnostic):')
        print(big_gap[['sensor', 'site', 'region', 'season',
                       'KS_train_vs_aeronet']].round(3).to_string(index=False))
else:
    print('No AERONET data loaded — check AERONET_FULL_DIR.')

## 6. Summary — coverage verdict per stratum

Roll-up of §1 pair-count and §2 KS / p99 / decile gates.  A stratum is `OK`
when both `N ≥ SOFT_CAL_MIN_PAIRS` and `coverage_pass = True`.  Strata that
fail are exactly the ones whose §7.4.1 (α, β) should be treated with the
lowest confidence in §8.1.2.

In [ ]:
if diag.empty:
    print('No diagnostics produced — confirm Stage A2 grids are populated and try again.')
else:
    summary = counts.merge(
        diag[['sensor', 'region', 'season',
              'KS', 'p99_ratio', 'empty_deciles', 'coverage_pass']],
        on=['sensor', 'region', 'season'], how='left',
    )
    summary['n_pass'] = summary['n_train'] >= SOFT_CAL_MIN_PAIRS / max(TRAIN_STRIDE, 1)
    summary['verdict'] = np.where(
        ~summary['n_pass'], 'LOW-N (→ guard rail)',
        np.where(summary['coverage_pass'].fillna(False), 'OK', 'WEAK COVERAGE'),
    )
    print('§7.4.1 training-pair coverage verdict:')
    cols = ['sensor', 'region', 'season', 'n_train', 'n_heldout',
            'KS', 'p99_ratio', 'empty_deciles', 'verdict']
    print(summary[cols].round(3).to_string(index=False))
    print()
    print('Verdict tally:')
    print(summary['verdict'].value_counts().to_string())